In [1]:
import pickle

from pyboolnet.external.bnet2primes import bnet_file2primes

from boolmore.io.load import import_phenotypes, check_phenotypes
from boolmore.algo.inference import get_phenotype_prediction
from boolmore.eval.score import get_phenotype_scores
from boolmore.io.export import export_phenotype_results

In [2]:
# BNET = "../case_study/T_cell/generated_models/20260613_1/Tcell_4405_gen28.bnet"
# OUTPUT_CACHE = "Tcell_4405_gen28_primes.pkl"

# INPUT_FILE = "../case_study/T_cell/Tcell_data.csv"

# OUTPUT_CSV = "Tcell_4405_gen28_results.csv"

In [3]:
# INPUT_CACHE = "../case_study/T_cell/Tcell_primes.pkl"

# INPUT_FILE = "../case_study/T_cell/Tcell_data.csv"
# OUTPUT_CSV = "Tcell_base_results.csv"

In [4]:
BNET = "../case_study/T_cell/N10/results/20260615_N10_consensus.bnet"

OUTPUT_CACHE = "20260615_N10_consensus.pkl"
INPUT_FILE = "../case_study/T_cell/N10/N10_data.csv"

OUTPUT_CSV = "20260615_N10_consensus.csv"

In [5]:
# with open(INPUT_CACHE, "rb") as f:
#     primes = pickle.load(f)
# print("Loaded primes from cache.")

primes = bnet_file2primes(BNET)
print("Loaded primes from bnet file.")
with open(OUTPUT_CACHE, "wb") as f:
    pickle.dump(primes, f)

Loaded primes from bnet file.


In [6]:
experiments = import_phenotypes(INPUT_FILE)

print(len(experiments))
for d in experiments:
    print(d)
    break

19
PhenotypeExperiment(id=10, perturbation=(), sources=(), phenotype=(('FOXP3', 0), ('GATA3', 0), ('IFNG', 0), ('IL17', 0), ('IL4', 0), ('RORGT', 0), ('TBET', 0), ('TGFB', 0)), expected_exists=True, weight=1.0)


In [7]:
check_phenotypes(primes, experiments)

In [8]:
import json
from boolmore.core.model import Model

json_file = "../case_study/T_cell/N10/N10_config.json"

f = open(json_file)
json_dict = json.load(f)

CONSTRAINTS = json_dict["constraints"]

model = Model.import_model(primes, constraints=CONSTRAINTS)

model.check_constraint()

True

In [9]:
print(experiments[1].perturbation)

def _assignment_to_dict(assignment):
    result = {}
    for node, value in assignment:
        result[node] = value
    return result

perturbation = _assignment_to_dict(experiments[1].perturbation)

print(perturbation)

()
{}


In [10]:
# from pyboolnet.prime_implicants import percolate

# # perc_primes = percolate(
# #     primes,
# #     add_constants=perturbation,
# #     remove_constants=False,
# #     copy=True,
# # )

# perc_primes = percolate(
#     primes,
#     add_constants={"IFNG":1},
#     copy=True
# )

# print(perc_primes)

In [11]:
predictions = get_phenotype_prediction(primes, experiments, debug=True)

for r in predictions:
    print(r)
    break


Experiment 10
  cache check: 0.000012 s
  get perc_primes: 0.000016 s
  compute max_trap: 0.018290 s

Experiment 11
  cache check: 0.000031 s
  cache hit (perc_primes)
  get perc_primes: 0.000029 s
  compute max_trap: 0.026382 s

Experiment 12
  cache check: 0.000038 s
  cache hit (perc_primes)
  get perc_primes: 0.000035 s
  compute max_trap: 0.018055 s

Experiment 13
  cache check: 0.000047 s
  cache hit (perc_primes)
  get perc_primes: 0.000032 s
  compute max_trap: 0.012698 s

Experiment 14
  cache check: 0.000036 s
  cache hit (perc_primes)
  get perc_primes: 0.000031 s
  compute max_trap: 0.012014 s

Experiment 15
  cache check: 0.000030 s
  cache hit (perc_primes)
  get perc_primes: 0.000014 s
  compute max_trap: 0.014102 s

Experiment 16
  cache check: 0.000036 s
  cache hit (perc_primes)
  get perc_primes: 0.000012 s
  compute max_trap: 0.015843 s

Experiment 17
  cache check: 0.000064 s
  cache hit (perc_primes)
  get perc_primes: 0.000031 s
  compute max_trap: 0.022752 s

E

In [12]:
score_items = get_phenotype_scores(experiments, predictions)

print(score_items[0])

EvaluationItemScore(id=10, weight=1.0, agreement=1.0, score=1.0)


In [13]:
export_phenotype_results(experiments, predictions, score_items, OUTPUT_CSV)

id | perturbation | sources                                                                                                                                                             | phenotype                                                                                             | expected_exists | predicted_exists | agreement | weight | score | found_phenotypes                                                                                                                                                                                                                                                                                                                                 
---+--------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------+-----------------

In [14]:
from boolmore.core.conversions import prime2bnet, prime2rr

GA_BNET = "../case_study/T_cell/N10/results/20260615/N10_704_gen12.bnet"

ga_primes = bnet_file2primes(GA_BNET)

print("\n-----comparing with the baseline functions-----")
modified = 0
for node in primes:
    if prime2rr(primes[node])[1] != prime2rr(ga_primes[node])[1]:
        modified += 1
        print("from_ga:" + prime2bnet(node, ga_primes[node]))
        print("from_consensus:" + prime2bnet(node, primes[node]))

print(f"\n{modified} out of {len(primes)} functions differ from the ga results")


-----comparing with the baseline functions-----
from_ga:GATA3,	STAT6 & !TBET | GATA3 & !TBET
from_consensus:GATA3,	GATA3 & !TBET
from_ga:IFNG,	!FOXP3 & NFAT & RUNX3 & !STAT3 & TBET & proliferation | !STAT3 & STAT4
from_consensus:IFNG,	!FOXP3 & NFAT & RUNX3 & !STAT3 & TBET & proliferation
from_ga:IL2RA,	NFAT & STAT5 | SMAD3 | NFKB
from_consensus:IL2RA,	NFAT & STAT5 | NFKB
from_ga:STAT3,	IL6R | IL27R | IL23R | IL21R | IL10R
from_consensus:STAT3,	IL6R | IL27R | IL21R
from_ga:STAT5,	STAT5 & STAT5_2 | IL4R | IL2R | IL15R
from_consensus:STAT5,	STAT5 & STAT5_2 | IL4R

5 out of 58 functions differ from the ga results
